# 07 Evaluation, tables and figures

Reproduces every table and figure of the results section from the per-hour forecasts
written by notebooks 01-06 into `outputs/`.

| Cell | Produces | Paper |
|---|---|---|
| 1 | paths, model list, file map, loaders | - |
| 2 | main results, both protocols | Table 3 (asymmetric) and the symmetric inputs to Table 4 |
| 3 | symmetric minus asymmetric | Table 4 |
| 4 | coverage-width comparison | Figure 3 |
| 5 | forecast waveforms, two test days | Figure 4 |
| 6 | dispersion of errors and widths | Figure 5 and the numbers in Section 4.6 |
| 7 | averages, percentages, and cost table | Table 5 and the aggregate numbers in the text |

Run the cells in order. Tables go to `outputs/tables/`, figures to `outputs/figures/`
(PDF for LaTeX and a PNG preview).

## 1. Setup

In [ ]:
# ============================================================
# 1. Setup: paths, model list, file map, loaders
# ============================================================
import os, sys, subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import ConnectionPatch, Patch

# REPO defaults to the parent of notebooks/ when run locally, or to
# MyDrive/uncertainty-placement on Colab. Override with SEGAN_REPO / SEGAN_OUT.
try:
    from google.colab import drive
    drive.mount("/content/drive")
    _default_repo = "/content/drive/MyDrive/uncertainty-placement"
except ImportError:
    _default_repo = os.path.abspath(os.path.join(os.getcwd(), ".."))
REPO     = os.environ.get("SEGAN_REPO", _default_repo)
OUT_ROOT = os.environ.get("SEGAN_OUT", os.path.join(REPO, "outputs"))
TAB_DIR  = os.path.join(OUT_ROOT, "tables")
FIG_DIR  = os.path.join(OUT_ROOT, "figures")
os.makedirs(TAB_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

MODELS    = ["SARIMAX", "XGBoost", "BiLSTM", "BiGRU-LSTM", "TFT", "LLM (ZT)", "LLM (FT)"]
PROTOCOLS = ["asymmetric", "symmetric"]
TARGET_COVERAGE = 0.80
TEST_DAYS = pd.DatetimeIndex(["2024-12-24", "2024-12-30"])

# (output folder, file stem) written by notebooks 01-06
FILES = {
    "asymmetric": {
        "SARIMAX":    ("sarimax_out_asymmetric_NATIVE_hourly",          "sarimax_asymmetric_native_hourly"),
        "XGBoost":    ("xgb_out_asymmetric_train5min_testhourly",       "xgb_asymmetric_hourly"),
        "BiLSTM":     ("bilstm_out_asymmetric_train5min_testhourly",    "bilstm_asymmetric_hourly"),
        "BiGRU-LSTM": ("bigrulstm_out_asymmetric_train5min_testhourly", "bigrulstm_asymmetric_hourly"),
        "TFT":        ("tft_out_asymmetric_FIXED_hourly",               "tft_asymmetric_hourly_FIXED"),
        "LLM (ZT)":   ("llm_out_asymmetric_hourly",                     "llm_zeroshot_asymmetric_hourly"),
        "LLM (FT)":   ("llm_out_asymmetric_hourly",                     "llm_trained_asymmetric_hourly"),
    },
    "symmetric": {
        "SARIMAX":    ("sarimax_out_symmetric_5min_route1_standardized_recursive", "sarimax_symmetric_5min"),
        "XGBoost":    ("xgb_out_symmetric_5min_route1_standardized",               "xgb_symmetric_5min"),
        "BiLSTM":     ("bilstm_out_symmetric_5min_fixed",                          "bilstm_symmetric_5min"),
        "BiGRU-LSTM": ("bigrulstm_out_symmetric_5min_route1_standardized",         "bigrulstm_symmetric_5min"),
        "TFT":        ("tft_out_symmetric_5min_recursive",                         "tft_symmetric_5min"),
        "LLM (ZT)":   ("llm_out_symmetric_5min_fixed1",                            "llm_zeroshot_symmetric_5min"),
        "LLM (FT)":   ("llm_out_symmetric_5min_fixed1",                            "llm_trained_symmetric_5min"),
    },
}

def paths(protocol, model):
    """(modular file, integrated file) for one model and protocol."""
    folder, stem = FILES[protocol][model]
    return (os.path.join(OUT_ROOT, folder, f"merged_route1_point_uq_series_{stem}.csv"),
            os.path.join(OUT_ROOT, folder, f"merged_route2_predictions_prob_{stem}.csv"))

def _load(path):
    d = pd.read_csv(path)
    d.columns = [c.strip() for c in d.columns]
    d = d.rename(columns={"Actual Power (W)": "actual", "q10 (W)": "q10",
                          "q50 (W)": "q50", "q90 (W)": "q90"})
    d["timestamp"] = pd.to_datetime(d["timestamp"], errors="coerce")
    if d["timestamp"].isna().any():
        raise ValueError(f"Unparsable timestamps in {path}")
    d["day"] = d["timestamp"].dt.normalize()
    return d.sort_values("timestamp").reset_index(drop=True)

def load_pair(protocol, model):
    """Modular (route 1: point_pred + quantiles) and integrated (route 2) forecasts."""
    p1, p2 = paths(protocol, model)
    return _load(p1), _load(p2)

def set_style(**extra):
    """Same 14-pt style for every figure; clears settings left by earlier cells."""
    plt.rcdefaults()
    plt.rcParams.update({"font.size": 14, "axes.titlesize": 14, "axes.labelsize": 14,
                         "xtick.labelsize": 14, "ytick.labelsize": 14, "legend.fontsize": 14,
                         "figure.dpi": 120, "savefig.bbox": "tight", **extra})

def save(fig, name):
    fig.savefig(os.path.join(FIG_DIR, f"{name}.pdf"))
    fig.savefig(os.path.join(FIG_DIR, f"{name}.png"), dpi=300)

missing = [p for pr in PROTOCOLS for m in MODELS for p in paths(pr, m) if not os.path.exists(p)]
print("outputs folder:", OUT_ROOT)
print("missing forecast files:", len(missing))
for p in missing:
    print("   ", p)


## 2. Main results (Table 3)

In [ ]:
# ============================================================
# 2. Main results (Table 3, and the symmetric table used for Table 4)
#    MAE      : modular uses the point forecast, integrated uses q0.5
#    Delta_cov: |PICP - 0.8| for the interval [q0.1, q0.9]
#    MPIW_80  : mean interval width
#    QS       : mean pinball loss over q0.1, q0.5, q0.9
# ============================================================
def pinball(y, qhat, q):
    e = y - qhat
    return np.maximum(q * e, (q - 1.0) * e)

def summarize(d, point_col):
    y, lo, mid, hi = (d[c].to_numpy(float) for c in ("actual", "q10", "q50", "q90"))
    inside = (y >= lo) & (y <= hi)
    qs = (pinball(y, lo, 0.1) + pinball(y, mid, 0.5) + pinball(y, hi, 0.9)) / 3.0
    return {"MAE": float(np.mean(np.abs(y - d[point_col].to_numpy(float)))),
            "Delta_cov": abs(float(inside.mean()) - TARGET_COVERAGE),
            "MPIW_80": float(np.mean(hi - lo)),
            "QS": float(np.mean(qs))}

def bold_pair(a, b, dec):
    best = min(a, b)
    fmt = lambda v: (r"\textbf{%.*f}" % (dec, v)) if np.isclose(v, best) else ("%.*f" % (dec, v))
    return fmt(a), fmt(b)

tables = {}
for protocol in PROTOCOLS:
    rows = []
    for m in MODELS:
        r1, r2 = load_pair(protocol, m)
        rows.append({"Model": m, "Formulation": "Modular",    **summarize(r1, "point_pred")})
        rows.append({"Model": m, "Formulation": "Integrated", **summarize(r2, "q50")})
    df = pd.DataFrame(rows)
    tables[protocol] = df
    df.to_csv(os.path.join(TAB_DIR, f"main_results_table_{protocol}.csv"), index=False)
    print(f"\n=== {protocol} ===")
    print(df.round(3).to_string(index=False))
    print("\nLaTeX rows:")
    for m in MODELS:
        a, b = (df[(df.Model == m) & (df.Formulation == f)].iloc[0] for f in ("Modular", "Integrated"))
        cells = [bold_pair(a[c], b[c], 3 if c == "Delta_cov" else 2) for c in ("MAE", "Delta_cov", "MPIW_80", "QS")]
        print(f"{m} & Modular & " + " & ".join(x[0] for x in cells) + r" \\")
        print(" & Integrated & " + " & ".join(x[1] for x in cells) + r" \\\hline")


## 3. Effect of removing the asymmetry (Table 4)

In [ ]:
# ============================================================
# 3. Effect of removing the asymmetry (Table 4): symmetric minus asymmetric
#    Negative values are improvements (green cells in the paper).
# ============================================================
key = ["Model", "Formulation"]
delta = (tables["symmetric"].set_index(key) - tables["asymmetric"].set_index(key)).reset_index()
delta.columns = key + ["Delta_MAE", "Delta_Delta_cov", "Delta_MPIW_80", "Delta_QS"]
delta.to_csv(os.path.join(TAB_DIR, "delta_results_table.csv"), index=False)
print(delta.round(3).to_string(index=False))

def cell(v, dec):
    s = f"{v:+.{dec}f}"
    return r"\cellcolor{green!70}" + s if v < 0 else s

print("\nLaTeX rows:")
for m in MODELS:
    for f, lead in (("Modular", m), ("Integrated", "")):
        r = delta[(delta.Model == m) & (delta.Formulation == f)].iloc[0]
        vals = [cell(r.Delta_MAE, 2), cell(r.Delta_Delta_cov, 3), cell(r.Delta_MPIW_80, 2), cell(r.Delta_QS, 2)]
        end = r" \\\hline" if f == "Integrated" else r" \\"
        print(f"{lead} & {f} & " + " & ".join(vals) + end)


## 4. Coverage-width comparison (Figure 3)

In [ ]:
# ============================================================
# 4. Coverage-width comparison (Figure 3)
#    (a) asymmetric, (b) symmetric; arrows link the two positions of each
#    model/placement. Colour = model, shape = placement.
# ============================================================
def coverage_width(protocol):
    rows = []
    for m in MODELS:
        for d, form in zip(load_pair(protocol, m), ("Modular U", "Integrated U")):
            inside = (d["actual"] >= d["q10"]) & (d["actual"] <= d["q90"])
            rows.append({"Backbone": m, "Formulation": form,
                         "PICP": float(inside.mean()), "MPIW": float((d["q90"] - d["q10"]).mean())})
    return pd.DataFrame(rows)

cw = {p: coverage_width(p) for p in PROTOCOLS}
for p in PROTOCOLS:
    print(f"{p}:\n", cw[p].round(3).to_string(index=False), "\n")

set_style(**{"font.size": 20, "axes.labelsize": 22, "axes.titlesize": 22,
             "xtick.labelsize": 18, "ytick.labelsize": 18, "legend.fontsize": 15,
             "legend.title_fontsize": 16, "axes.linewidth": 1.6})
color_map  = {m: plt.cm.tab10.colors[i] for i, m in enumerate(MODELS)}
marker_map = {"Modular U": "^", "Integrated U": "s"}

allx = np.concatenate([cw[p]["MPIW"] for p in PROTOCOLS])
ally = np.concatenate([cw[p]["PICP"] for p in PROTOCOLS])
xpad, ypad = 0.03 * (allx.max() - allx.min()), 0.04 * (ally.max() - ally.min())
x_lim = (allx.min() - xpad, allx.max() + xpad)
y_lim = (max(0, ally.min() - ypad), min(1, ally.max() + ypad))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 7.5), sharex=True, sharey=True)
coords = {}
for ax, p, title in ((ax1, "asymmetric", "(a) Asymmetric"), (ax2, "symmetric", "(b) Symmetric")):
    ax.set_xlim(*x_lim); ax.set_ylim(*y_lim)
    ax.axhline(TARGET_COVERAGE, linestyle="--", linewidth=2.0, color="black")
    ax.grid(True, linestyle="--", linewidth=1.0, color="black", alpha=0.2)
    ax.set_xlabel(r"$\mathrm{MPIW}_{80}$ (W)")
    ax.set_title(title)
    ideal_x, ideal_y = x_lim[0] + 0.35 * (x_lim[1] - x_lim[0]), 0.67
    ax.fill_betweenx([ideal_y, y_lim[1]], x_lim[0], ideal_x, color="black", alpha=0.06, zorder=0)
    ax.text(x_lim[0] + 0.015 * (x_lim[1] - x_lim[0]), ideal_y + 0.012,
            "Desired region", fontsize=16, va="bottom", zorder=2)
    for _, r in cw[p].iterrows():
        coords[(p, r.Backbone, r.Formulation)] = (r.MPIW, r.PICP)
        ax.scatter(r.MPIW, r.PICP, s=300, marker=marker_map[r.Formulation],
                   facecolor=color_map[r.Backbone], edgecolor="black", linewidth=1.3, zorder=4)
ax1.set_ylabel("PICP")

for m in MODELS:                       # arrows: symmetric position -> asymmetric position
    for form in marker_map:
        fig.add_artist(ConnectionPatch(
            xyA=coords[("symmetric", m, form)],  coordsA=ax2.transData,
            xyB=coords[("asymmetric", m, form)], coordsB=ax1.transData,
            arrowstyle="->", lw=0.9, color="black", alpha=0.6, mutation_scale=20))

model_handles = [Line2D([0], [0], marker="o", linestyle="None", markersize=13,
                        markerfacecolor=color_map[m], markeredgecolor="black", label=m) for m in MODELS]
form_handles = [Line2D([0], [0], marker=mk, linestyle="None", markersize=15, markerfacecolor="white",
                       markeredgecolor="black", markeredgewidth=1.5, label=f) for f, mk in marker_map.items()]
leg = ax2.legend(handles=form_handles, loc="lower center", title="Placement (shape)", framealpha=0.95)
ax2.add_artist(leg)
ax2.legend(handles=model_handles, loc="lower right", title="Model (colour)", framealpha=0.95)
fig.tight_layout()
save(fig, "fig3_coverage_width")      # paper file: 5.pdf
plt.show()


## 5. Forecast waveforms (Figure 4)

In [ ]:
# ============================================================
# 5. Forecast waveforms over the two test days (Figure 4)
#    Actual, point forecast, modular band (orange), integrated band (green).
# ============================================================
set_style()
FIG4 = {"XGBoost": ("a", "b"), "BiGRU-LSTM": ("c", "d"), "LLM (FT)": ("e", "f")}   # panels of Figure 4
COLOR_ACTUAL, COLOR_POINT = "#2c7bb6", "#d7191c"
COLOR_MODULAR, COLOR_INTEGRATED = "#fdae61", "#1a9641"

def plot_back_to_back(protocol, model, panel):
    r1, r2 = load_pair(protocol, model)
    fig = plt.figure(figsize=(11, 4.3))
    plt.title(f"{model} - {protocol.capitalize()} - Back-to-Back Days")
    plt.ylabel("Power (W)")
    x0, ticks, labels, used = 0, [], [], set()
    lab = lambda name: None if name in used else (used.add(name) or name)
    for day in TEST_DAYS:
        a, b = r1[r1["day"] == day], r2[r2["day"] == day]
        n = max(len(a), len(b))
        if n == 0:
            continue
        xa, xb = np.arange(len(a)) + x0, np.arange(len(b)) + x0
        plt.plot(xa, a["actual"], linewidth=2, color=COLOR_ACTUAL, label=lab("Actual"))
        plt.plot(xa, a["point_pred"], "--", linewidth=1.8, color=COLOR_POINT, label=lab("Point Prediction"))
        plt.fill_between(xa, a["q10"], a["q90"], alpha=0.18, color=COLOR_MODULAR, label=lab("Modular U: 10-90%"))
        plt.plot(xa, a["q50"], linewidth=1.3, color=COLOR_MODULAR, label=lab("Modular U (q50)"))
        plt.fill_between(xb, b["q10"], b["q90"], alpha=0.18, color=COLOR_INTEGRATED, label=lab("Integrated U: 10-90%"))
        plt.plot(xb, b["q50"], linewidth=1.3, color=COLOR_INTEGRATED, label=lab("Integrated U (q50)"))
        if x0 > 0:
            plt.axvline(x0 - 0.5, linestyle="--", alpha=0.5)
        ticks.append(x0 + (n - 1) / 2); labels.append(day.strftime("%b %d")); x0 += n
    plt.xticks(ticks, labels); plt.xlabel("Days")
    plt.grid(True, linestyle="--", alpha=0.5)
    plt.legend(ncol=2, frameon=False, loc="upper left")
    save(fig, f"fig4{panel}_{model.replace(' ', '').replace('(', '_').replace(')', '')}_{protocol}")
    plt.show()

for model, (pa, pb) in FIG4.items():
    plot_back_to_back("asymmetric", model, pa)
    plot_back_to_back("symmetric",  model, pb)


## 6. Dispersion of errors and widths (Figure 5)

In [ ]:
# ============================================================
# 6. Dispersion of hourly errors and interval widths (Figure 5, Section 4.6)
#    Colour = model (as in Figure 3); solid = modular, hatched = integrated;
#    dark = asymmetric, light = symmetric.
# ============================================================
rows = []
for p in PROTOCOLS:
    for m in MODELS:
        for d, pl in zip(load_pair(p, m), ("Modular", "Integrated")):
            rows.append(pd.DataFrame({"protocol": p.capitalize(), "model": m, "placement": pl,
                                      "abs_err": (d["actual"] - d["q50"]).abs(),
                                      "width": d["q90"] - d["q10"]}))
disp = pd.concat(rows, ignore_index=True)

set_style(**{"xtick.labelsize": 14, "legend.fontsize": 13})
color_map = {m: plt.cm.tab10.colors[i] for i, m in enumerate(MODELS)}
RUNS  = [("Asymmetric", "Modular"), ("Asymmetric", "Integrated"), ("Symmetric", "Modular"), ("Symmetric", "Integrated")]
ALPHA = {"Asymmetric": 0.9, "Symmetric": 0.35}
HATCH = {"Modular": None, "Integrated": "///"}

fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
for ax, col, ylab in ((axes[0], "abs_err", "Absolute error (W)"), (axes[1], "width", "80% interval width (W)")):
    for i, m in enumerate(MODELS):
        for k, (prot, pl) in enumerate(RUNS):
            vals = disp[(disp.protocol == prot) & (disp.placement == pl) & (disp.model == m)][col].values
            bp = ax.boxplot([vals], positions=[i * 5 + k], widths=0.8, patch_artist=True,
                            medianprops=dict(color="black", linewidth=1.4))
            bp["boxes"][0].set(facecolor=color_map[m], alpha=ALPHA[prot], edgecolor="black",
                               linewidth=1.0, hatch=HATCH[pl])
            for part in ("whiskers", "caps"):
                for line in bp[part]:
                    line.set(color="black", linewidth=1.0)
            for f in bp["fliers"]:
                f.set(marker=".", markersize=3, markeredgecolor=color_map[m], alpha=0.7)
        if i > 0:
            ax.axvline(i * 5 - 1, color="0.8", linewidth=0.8, zorder=0)
    ax.set_ylabel(ylab)
    ax.grid(True, axis="y", linestyle="--", alpha=0.5)
axes[1].set_xticks([i * 5 + 1.5 for i in range(len(MODELS))])
axes[1].set_xticklabels(MODELS)
handles = [Patch(facecolor="0.4", alpha=ALPHA[p], edgecolor="black", hatch=HATCH[pl], label=f"{pl} U - {p}")
           for p, pl in RUNS]
axes[0].legend(handles=handles, ncol=2, loc="lower center", bbox_to_anchor=(0.5, 1.02),
               frameon=False, columnspacing=1.5, handlelength=2.2)
fig.tight_layout()
save(fig, "fig5_dispersion_boxplots")   # paper file: 9.pdf
plt.show()

summary = disp.groupby(["protocol", "model", "placement"]).agg(
    err_median=("abs_err", "median"),
    err_iqr=("abs_err", lambda x: x.quantile(.75) - x.quantile(.25)),
    err_p95=("abs_err", lambda x: x.quantile(.95)),
    width_median=("width", "median"),
    width_iqr=("width", lambda x: x.quantile(.75) - x.quantile(.25))).round(1)
summary.to_csv(os.path.join(TAB_DIR, "dispersion_summary.csv"))
print(summary.to_string())


## 7. Aggregate numbers and cost table (Table 5)

In [ ]:
# ============================================================
# 7. Aggregate numbers and cost table (Table 5)
#    Runs scripts/paper_numbers.py on the tables from cell 2. It rebuilds the
#    cost table and checks every average, percentage, and cost quoted in the paper.
# ============================================================
env = dict(os.environ, SEGAN_RESULTS=TAB_DIR)
res = subprocess.run([sys.executable, os.path.join(REPO, "scripts", "paper_numbers.py")],
                     env=env, capture_output=True, text=True)
print(res.stdout)
if res.returncode != 0:
    print(res.stderr or "Some quoted numbers differ from these tables (see above).")
